# NutriTrack API Test
Test script for the NutriTrack FastAPI server.

**Prerequisites**: Start the API server first:
```bash
cd d:\Project\Code\nutritrack-documentation\app
python templates/api.py
```

In [10]:
import requests
import json
import os

BASE_URL = "http://localhost:8000"
DATA_DIR = r"D:\Project\Code\nutritrack-documentation\data\images\food"

## 1. Health Check

In [11]:
res = requests.get(f"{BASE_URL}/health")
print(f"Status: {res.status_code}")
print(json.dumps(res.json(), indent=2))

Status: 200
{
  "status": "ok",
  "qwen_model": "qwen.qwen3-vl-235b-a22b",
  "usda_ready": true
}


## 2. Analyze Food Image — fast_food.jpg

In [12]:
img_path = r"D:\Project\Code\nutritrack-documentation\data\images\food\com_tam.jpg"
print(f"Uploading: {img_path}")
print(f"File size: {os.path.getsize(img_path) / 1024:.0f} KB")

with open(img_path, "rb") as f:
    res = requests.post(
        f"{BASE_URL}/analyze",
        files={"file": ("fast_food.jpg", f, "image/jpeg")}
    )

print(f"\nStatus: {res.status_code}")
data = res.json()
print(f"Message: {data.get('message')}")
print(f"Dishes found: {len(data.get('data', []))}")

Uploading: D:\Project\Code\nutritrack-documentation\data\images\food\com_tam.jpg
File size: 93 KB

Status: 200
Message: Phát hiện 1 món ăn.
Dishes found: 1


In [13]:
# Pretty print full results
for i, dish in enumerate(data["data"], 1):
    t = dish["total_nutrition"]
    print(f"\n{'='*60}")
    print(f"🍽️  {i}. {dish['name']} ({dish['vi_name']})")
    print(f"   AI Calories: {dish['qwen_estimated_calories']} kcal")
    print(f"   USDA Total → Cal: {t['calories']:.0f} | Pro: {t['protein']:.1f}g | Carb: {t['carbs']:.1f}g | Fat: {t['fat']:.1f}g")
    print(f"   Avg Calories: {dish['average_calories']:.0f}")
    print(f"   Ingredients ({len(dish['ingredients'])}):\n")
    for ing in dish["ingredients"]:
        n = ing["nutrition"]
        print(f"     - {ing['name']:20s} {ing['weight_g']:6.0f}g  Cal:{n['calories']:5.0f}  P:{n['protein']:4.1f}  C:{n['carbs']:4.1f}  F:{n['fat']:4.1f}")


🍽️  1. Vietnamese Pork Chop Rice (Cơm Tấm Sườn Nướng)
   AI Calories: 750.0 kcal
   USDA Total → Cal: 1219 | Pro: 82.6g | Carb: 140.9g | Fat: 37.4g
   Avg Calories: 985
   Ingredients (10):

     - Grilled pork chop       180g  Cal:  342  P:33.1  C:14.1  F:16.3
     - Broken rice             150g  Cal:  466  P:20.0  C:100.1  F: 1.7
     - Fried egg                60g  Cal:  118  P: 8.2  C: 0.5  F: 8.9
     - Pork meatloaf            70g  Cal:  152  P: 8.2  C:20.1  F: 4.3
     - Pickled carrots and daikon     40g  Cal:   15  P: 0.0  C: 3.1  F: 0.0
     - Tomato                   30g  Cal:    0  P: 0.0  C: 0.0  F: 0.0
     - Cucumber                 25g  Cal:    8  P: 0.1  C: 1.7  F: 0.0
     - Scallion oil              5g  Cal:    2  P: 0.1  C: 0.4  F: 0.0
     - Pork skin (crispy)       20g  Cal:  109  P:12.3  C: 0.0  F: 6.3
     - Fresh herb salad         30g  Cal:    7  P: 0.7  C: 1.1  F: 0.0


## 3. Analyze Food Image — com_tam.jpg

In [ ]:
img_path = os.path.join(DATA_DIR, "com_tam.jpg")
print(f"Uploading: {img_path}")

with open(img_path, "rb") as f:
    res = requests.post(
        f"{BASE_URL}/analyze",
        files={"file": ("com_tam.jpg", f, "image/jpeg")}
    )

data = res.json()
print(f"Status: {res.status_code} | {data.get('message')}")

for i, dish in enumerate(data["data"], 1):
    t = dish["total_nutrition"]
    print(f"\n🍽️  {i}. {dish['name']} ({dish['vi_name']})")
    print(f"   Cal: {t['calories']:.0f} | Pro: {t['protein']:.1f}g | Carb: {t['carbs']:.1f}g | Fat: {t['fat']:.1f}g")
    for ing in dish["ingredients"]:
        print(f"     - {ing['name']:20s} {ing['weight_g']:5.0f}g")

## 4. Error Handling — Invalid File

In [14]:
# Test with non-image file
res = requests.post(
    f"{BASE_URL}/analyze",
    files={"file": ("test.txt", b"not an image", "text/plain")}
)
print(f"Status: {res.status_code}")
print(f"Response: {res.json()}")
assert res.status_code == 400, "Should reject non-image files"
print("✅ Error handling works correctly!")

Status: 400
Response: {'detail': 'File must be an image'}
✅ Error handling works correctly!


Lỗi: Analysis failed: An error occurred
(ValidationException) when calling the Converse
operation: The model returned the following
errors: ("error":
{"code": "validation error" "message":"Failed to
buffer the request body: length limit
exceeded","param":null,"type". *invalid_request_error

Lối: The string did not match the expected
pattern.